# Build Graph NPZ từ PCAP thô trên Kaggle

## Cần tải lên Kaggle trước khi chạy

### Bắt buộc
| Dataset Kaggle | File | Nguồn trong repo |
|---|---|---|
| `nt114-pcap-dataset` | `raw.rar` | `data/raw.rar` |

### Rất nên có – tiết kiệm 3–6 giờ compute
| Dataset Kaggle | File | Nguồn trong repo |
|---|---|---|
| `nt114-student-model` | `student_cnn_best.pt` | `outputs/student_cnn/student_cnn_best.pt` |

### Nên có – tiết kiệm ~30 phút (bỏ qua MITRE embedding generation)
| Dataset Kaggle | File | Nguồn trong repo |
|---|---|---|
| `nt114-mitre` | `mitre_techniques.csv` | `data/mitre/mitre_techniques.csv` |
| | `mitre_tactics.csv` | `data/mitre/mitre_tactics.csv` |
| | `mitre_technique_tactic_edges.csv` | `data/mitre/mitre_technique_tactic_edges.csv` |
| | `mitre_techniques_embeddings.npy` | `data/mitre/mitre_techniques_embeddings.npy` |

> MITRE data không phụ thuộc vào dataset PCAP — file trong `data/mitre/` vẫn dùng được.

### Tùy chọn – tránh download ~400 MB SecureBERT từ HuggingFace
| Dataset Kaggle | Nội dung |
|---|---|
| `securebert` | Folder model SecureBERT (chỉ cần nếu thiếu `mitre_techniques_embeddings.npy`) |

---

**Luồng xử lý:**
```
raw.rar → giải nén PCAP → extract payload_256.npy + metadata.csv
        → (bỏ qua teacher + training nếu có student_cnn_best.pt)
        → export student embeddings  [GPU]
        → build/load MITRE knowledge base
        → build three-tier graph NPZ  [GPU batched cosine similarity]
        → graph_npz_artifact.zip
```

In [ ]:
from __future__ import annotations

import shutil
import subprocess
import sys
from pathlib import Path

# ── Repo ──────────────────────────────────────────────────────────────────────
GITHUB_REPO_URL = "https://github.com/LeThanhPhat-ATTT2023/Do-an-chuyen-nganh_NT114.git"
GITHUB_BRANCH   = "main"
FORCE_RECLONE   = False

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_DIR   = Path("/kaggle/working/Do-an-chuyen-nganh_NT114")
WORK_DIR   = Path("/kaggle/working/nt114_npz_work")
OUTPUT_ZIP = Path("/kaggle/working/graph_npz_artifact.zip")

# ── Payload extraction ─────────────────────────────────────────────────────────
RESET_WORK_DIR          = False
PAYLOAD_EXTRACT_WORKERS = 0       # 0 = tự động theo số CPU
LOG_EVERY_PACKETS       = 100_000
WRITE_BATCH_SIZE        = 100_000
MAX_PACKETS_PER_FILE    = None    # đặt int (vd: 5000) để chạy test nhanh

# ── Embedding model ────────────────────────────────────────────────────────────
# Nếu upload folder SecureBERT lên Kaggle thì sẽ tự detect.
# Nếu không có và thiếu mitre_techniques_embeddings.npy, sẽ download tự động.
TEACHER_MODEL_NAME        = "ehsanaghaei/SecureBERT"
EMBEDDING_MODEL_PATH      = ""    # ghi đường dẫn cụ thể nếu cần override
AUTO_FIND_EMBEDDING_MODEL = True

# ── Student CNN ────────────────────────────────────────────────────────────────
# Chỉ dùng nếu KHÔNG có student_cnn_best.pt trong /kaggle/input.
TEACHER_BATCH_SIZE     = 32
STUDENT_EPOCHS         = 30
STUDENT_BATCH_SIZE     = 256
STUDENT_NUM_WORKERS    = 2
STUDENT_EMB_BATCH_SIZE = 1024
MITRE_BATCH_SIZE       = 64
DEVICE                 = "auto"   # auto → dùng GPU nếu có, fallback CPU

# ── Graph builder ─────────────────────────────────────────────────────────────
SIMILARITY_THRESHOLD = 0.82
PACKET_TOP_K         = 5
FLOW_TOP_K           = 5
# Packets per GPU/CPU batch khi tính cosine similarity.
# T4 16 GB: 50_000 an toàn.  Nếu OOM thì giảm xuống 20_000.
SIM_BATCH_SIZE       = 50_000

In [ ]:
# Kiểm tra những gì đã upload vào /kaggle/input
import torch
INPUT_ROOT = Path("/kaggle/input")

print("=== GPU info ===")
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory / 1024**3:.1f} GB VRAM")

print()
print("=== /kaggle/input – artifacts ===")
CHECK_FILES = [
    "raw.rar",
    "student_cnn_best.pt",
    "mitre_techniques.csv",
    "mitre_tactics.csv",
    "mitre_technique_tactic_edges.csv",
    "mitre_techniques_embeddings.npy",
    "teacher_targets.npy",
    "student_embeddings.npy",
]
for name in CHECK_FILES:
    matches = sorted(INPUT_ROOT.glob(f"**/{name}")) if INPUT_ROOT.exists() else []
    tag    = "[OK]     " if matches else "[MISSING]"
    detail = str(matches[0]) if matches else "(chưa upload)"
    print(f"  {tag} {name:<45} {detail}")

print()
print("=== Datasets trong /kaggle/input ===")
if INPUT_ROOT.exists():
    for p in sorted(INPUT_ROOT.iterdir()):
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"  {p.name:<40} ({n} files)")

In [ ]:
def run(cmd: list[str], cwd: Path | None = None) -> None:
    print("\n$", " ".join(str(x) for x in cmd), flush=True)
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)


if FORCE_RECLONE and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    clone_cmd = ["git", "clone"]
    if GITHUB_BRANCH:
        clone_cmd += ["--branch", GITHUB_BRANCH]
    clone_cmd += [GITHUB_REPO_URL, str(REPO_DIR)]
    run(clone_cmd)
else:
    print("Repo đã tồn tại, pull mới nhất...")
    run(["git", "-C", str(REPO_DIR), "pull"])

run([
    sys.executable, "-m", "pip", "install", "-q",
    "scapy>=2.5.0", "transformers>=4.40", "sentencepiece", "accelerate",
])
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

script = REPO_DIR / "notebooks" / "build_graph_npz_from_zipped_pcaps_kaggle.py"
assert script.exists(), f"Driver script không tìm thấy: {script}"
print("Driver script:", script)

In [ ]:
script = REPO_DIR / "notebooks" / "build_graph_npz_from_zipped_pcaps_kaggle.py"

cmd = [
    sys.executable, "-u", str(script),
    "--input-root",              "/kaggle/input",
    "--work-dir",                 str(WORK_DIR),
    "--output-zip",               str(OUTPUT_ZIP),
    "--payload-extract-workers",  str(PAYLOAD_EXTRACT_WORKERS),
    "--log-every-packets",        str(LOG_EVERY_PACKETS),
    "--write-batch-size",         str(WRITE_BATCH_SIZE),
    "--teacher-model-name",       TEACHER_MODEL_NAME,
    "--teacher-batch-size",       str(TEACHER_BATCH_SIZE),
    "--student-epochs",           str(STUDENT_EPOCHS),
    "--student-batch-size",       str(STUDENT_BATCH_SIZE),
    "--student-num-workers",      str(STUDENT_NUM_WORKERS),
    "--student-emb-batch-size",   str(STUDENT_EMB_BATCH_SIZE),
    "--mitre-batch-size",         str(MITRE_BATCH_SIZE),
    "--device",                   DEVICE,
    "--similarity-threshold",     str(SIMILARITY_THRESHOLD),
    "--packet-top-k",             str(PACKET_TOP_K),
    "--flow-top-k",               str(FLOW_TOP_K),
    "--sim-batch-size",           str(SIM_BATCH_SIZE),
]

if RESET_WORK_DIR:
    cmd.append("--reset-work-dir")
if MAX_PACKETS_PER_FILE is not None:
    cmd += ["--max-packets-per-file", str(MAX_PACKETS_PER_FILE)]
if EMBEDDING_MODEL_PATH:
    cmd += ["--embedding-model-path", EMBEDDING_MODEL_PATH]
if not AUTO_FIND_EMBEDDING_MODEL:
    cmd += ["--no-auto-find-embedding-model"]

run(cmd, cwd=REPO_DIR)

In [ ]:
import json

if not OUTPUT_ZIP.exists():
    raise FileNotFoundError(f"Output zip không tồn tại: {OUTPUT_ZIP}")

print("Download:", OUTPUT_ZIP)
print("Size GB: ", round(OUTPUT_ZIP.stat().st_size / 1024**3, 3))

print("\nGraph artifacts:")
processed = WORK_DIR / "data" / "processed"
for path in sorted(processed.glob("graph_artifact_3tier_t082_k5*")):
    print(" -", path, round(path.stat().st_size / 1024**3, 3), "GB")

meta_path = processed / "graph_artifact_3tier_t082_k5.meta.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    keys = ["num_packets", "num_flows", "num_techniques", "num_tactics",
            "num_packet_technique_edges", "num_flow_technique_edges",
            "device", "sim_batch_size"]
    print("\nGraph stats:")
    print(json.dumps({k: meta.get(k) for k in keys}, indent=2))